In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Mapping data from Gotze et al

In [15]:
#Read the data
el_data = pd.read_excel('../data/external/elemental_data_gotze.xlsx', sheet_name='all Means')
el_data_std = pd.read_excel('../data/external/elemental_data_gotze.xlsx', sheet_name='all STDEV')
sample_data = pd.read_csv('../data/external/country_sample.csv')

The groupings are different for each stream. 

Food Waste:  We cannot group the entire organic waste actegory, so we group it by 'Material type' - food waste and gardening waste
Glass, metal, paper, plastic: they are their own category (in material type) as 'textiles, leather, rubber', 'wood'
textile and wood are specfic fractions within combustibles
There is no bio plastic category
OTHER is not clear!! - other from combustibles/inert? or all residual waste?


In [ ]:
#Melt the table
units = el_data.iloc[0]
el_data.drop(0, inplace=True)
el_data = el_data.melt(id_vars=['Fraction ID', 'Material type', 'Fraction name', 'Waste flow'], var_name='Physiochemical property', value_name='Value')
el_data['Unit'] = el_data['Physiochemical property'].map(units)

In [42]:
el_data.loc[el_data['Value'].apply(type) == str,'Value']=np.nan

In [30]:
# Group by for textile and wood
text_wood = el_data.loc[el_data['Fraction name'].isin(['textiles, leather, rubber', 'wood'])].groupby(['Fraction ID','Material type','Fraction name', 'Waste flow', 'Physiochemical property', 'Unit'], as_index = False)['Value'].agg(['min','max'])


In [55]:

rest = el_data.loc[(~el_data['Material type'].isin(['combustibles', 'inert ']))].groupby(['Material type', 'Waste flow', 'Physiochemical property', 'Unit'], as_index = False)['Value'].agg(['min','max'])

In [ ]:
# reshaping and concatenating the dataframes
text_wood.drop(['Fraction ID','Material type'], inplace=True, axis=1)
text_wood.rename(columns={'Fraction name':'Material type'}, inplace=True)


In [57]:
el_data_mapped = pd.concat([rest, text_wood]).reset_index(drop=True)

In [59]:
el_data_mapped['Material type'].unique()

array(['food waste', 'gardening waste', 'glass', 'metal',
       'paper & cardboard', 'plastic', 'textiles, leather, rubber',
       'wood'], dtype=object)

In [60]:
code_mapping={
    'bio plastic':'MSW_URB_BIO_PLA', 
    'food waste': 'MSW_URB_FOOD', 
    'glass': 'MSW_URB_GLA', 
    'metal': 'MSW_URB_MET',
    'other': 'MSW_URB_OTH',
    'paper & cardboard': 'MSW_URB_PAP', 
    'plastic': 'MSW_URB_PLA', 
    'textiles, leather, rubber': 'MSW_URB_TEX',
    'wood': 'MSW_URB_WOOD'
}

In [61]:
el_data_mapped['sector'] = el_data_mapped['Material type'].map(code_mapping)

In [ ]:
#el_data_mapped.to_csv('../data/processed/elemental_data_gotze.csv', index=False)

In [66]:
el_data_mapped['diff']=(el_data_mapped['max']-el_data_mapped['min'])/el_data_mapped['min'].replace(0, np.nan)

/var/folders/bm/tmx1rxns5yvdzj8zkylvh57xfl_lm7/T/ipykernel_17320/2427470581.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  el_data_mapped['diff']=(el_data_mapped['max']-el_data_mapped['min'])/el_data_mapped['min'].replace(0, np.nan)


In [70]:
a = el_data_mapped.sort_values('diff', ascending=False)

In [74]:
display(a)

,Material type,Waste flow,Physiochemical property,Unit,min,max,sector,diff
374,metal,residual,VS,%TS,0.053,63.06,MSW_URB_MET,1188.811321
319,metal,residual,Cu,mg/kgTS,68.8,45704.9,MSW_URB_MET,663.315407
695,plastic,source-segegated,Cl,%TS,0.00377,2.4516,MSW_URB_PLA,649.291777
663,plastic,residual,Sb,mg/kgTS,0.453,270.889,MSW_URB_PLA,596.988962
739,plastic,source-segegated,Sb,mg/kgTS,0.433,254.451,MSW_URB_PLA,586.646651
...,...,...,...,...,...,...,...,...
450,metal,source-segegated,VS,%TS,-3.927,0.7267,MSW_URB_MET,-1.185052
576,paper & cardboard,source-segegated,O,%TS,NaN,NaN,MSW_URB_PAP,NaN
665,plastic,residual,Se,mg/kgTS,0.0,3.01407,MSW_URB_PLA,NaN
728,plastic,source-segegated,O,%TS,NaN,NaN,MSW_URB_PLA,NaN


In [76]:
el_data.loc[el_data['Material type']=='metal']['Fraction name'].unique()


array(['metal packaging-non-ferrous', 'metal packaging-ferrous',
       'aluminium foil', 'non-packaging metal-ferrous',
       'non-packaging metal-non-ferrous'], dtype=object)